> 📖 **參考解答**：請先在學生版自己試，卡住再對照這份。學生版在 [`notebooks/`](../notebooks/)，這份與學生版的差別只在練習題的空格（`____`）已填好。

# PBAIH11-A 實作一：從 mzML 原始檔到特徵表（參考解答）

**目標**：把 12 個 LC-MS 原始檔（mzML）變成一張「特徵 × 樣本」的強度矩陣，也就是講課中流程的 ①原始檔 → ②峰偵測 → ③對齊 → ④特徵表。我們接續 PBAIH07 用過的 **pyopenms**。

**資料**：Sacurine toy set（人類尿液、LTQ-Orbitrap、ESI 負離子模式）。12 個檔：6 個受試者樣本（`HU_neg_*`）、3 個 pooled QC（`QC1_*`）、3 個 blank（`Blanc*`）。
來源：Zenodo record 10130758（CC-BY-4.0；原研究為 MetaboLights MTBLS404）。為了讓 Colab 跑得快，課程版已**裁切**成 m/z 70–600、RT 60–1000 秒（zip 約 46 MB，解壓後約 115 MB）。

| 部分 | 內容 | 建議時間 |
|---|---|---|
| 核心 0 | 環境與資料（**上課一開始先執行**） | 上課一開始 |
| 核心 1 | 讀一個 mzML、看 TIC／BPC、確認 centroid（練習 1） | 5 分 |
| 核心 2 | 單一檔案特徵偵測、畫一個特徵的 EIC（練習 2） | 8 分 |
| 核心 3 | 一次跑完 12 個檔、調參數看特徵數（練習 3） | 4 分 |
| 核心 4 | 跨樣本對應 → 特徵表、看缺值 | 4 分 |
| 核心 5 | blank 過濾（練習 4） | 4 分 |
| 延伸 E1–E3 | 保留時間對齊、QC RSD 預覽、存 CSV 與參數敏感度（**課後／有餘力**） | 課後 |

每一節都分成：**為什麼**（這步要解決什麼問題）→ **怎麼做**（程式）→ **看結果** → **想一想**。練習題是「補完這一行」，空格寫成 `____`。

🆘 **卡住時**：① 先讀錯誤訊息的最後一行（通常會說是哪個變數或哪一行）→ ② 問 Colab 內建的 Gemini（把錯誤訊息貼給它，請它「用中文解釋」）→ ③ 打開下方的 💡 提示。

## 執行環境

本 notebook 以 Google Colab 為準（Python 3.12）。實測版本：

| 套件 | 版本 | 備註 |
|---|---|---|
| pyopenms | 3.6.0 | Colab 沒有內建，第 0 節會安裝；**3.6 版的 API 和舊教學不同**（例如 `MassTraceDetection.run()` 直接回傳 list） |
| pandas | 2.2.3 | Colab 內建 |
| scipy | 1.16.3 | Colab 內建 |
| scikit-learn | 1.6.1 | Colab 內建（實作二才用） |
| matplotlib | 3.10.0 | Colab 內建 |
| seaborn | 0.13.2 | Colab 內建 |

圖表的文字用英文，因為 Colab 預設沒有中文字型。

---
# 核心路徑（分兩節：第 0–2 節在第 1 節後段，第 3–5 節在第 2 節前段）

## 0. 環境與資料（上課一開始先執行）

**為什麼**：安裝 pyopenms 和下載資料（zip 約 46 MB）是整份實作最慢的部分，需要等一下，請在上課一開始就先執行下面這一格，趁它在跑的時候聽老師開場。

**怎麼做**：執行下面這一格。它會（1）在 Colab 上安裝 `pyopenms==3.6.0`；（2）從課程的 GitHub（Releases）下載資料並解壓縮；（3）印出套件版本。已經下載過就不會重複下載。

> 開發／本機執行：若已有資料，可先設定環境變數 `PBAIH11_DATA_DIR` 指向放 mzML 的資料夾，這一格就會略過下載。

In [ ]:
# 0. 安裝套件、下載資料、印出版本（上課一開始先執行這一格）
import os, sys, glob, time, zipfile

IN_COLAB = "google.colab" in sys.modules

# (1) 安裝 pyopenms：只在 Colab 上安裝（等同於 %pip install pyopenms==3.6.0）
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q pyopenms==3.6.0")

# (2) 取得資料：優先順序 = 環境變數 PBAIH11_DATA_DIR → 已下載的資料夾 → 從課程 GitHub 下載
DATA_URL = "https://github.com/wang3yuan/PBAIH/releases/download/data-v1/sacurine_cropped.zip"
DATA_ROOT = "pbaih11_data"

def find_data_dir(root):
    """在 root 底下找第一個含有 .mzML 的資料夾。"""
    hits = sorted(glob.glob(os.path.join(root, "**", "*.mzML"), recursive=True))
    return os.path.dirname(hits[0]) if hits else None

DATA_DIR = os.environ.get("PBAIH11_DATA_DIR") or find_data_dir(DATA_ROOT)
if DATA_DIR is None:
    import urllib.request
    os.makedirs(DATA_ROOT, exist_ok=True)
    zip_path = os.path.join(DATA_ROOT, "sacurine_cropped.zip")
    print("下載中（約 46 MB）...")
    urllib.request.urlretrieve(DATA_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_ROOT)
    DATA_DIR = find_data_dir(DATA_ROOT)

mzml_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.mzML")))
meta_files = glob.glob(os.path.join(DATA_DIR, "sampleMetadata*.tsv"))
print("資料夾：", DATA_DIR)
print("mzML 檔數：", len(mzml_files), "（應為 12）")
assert len(mzml_files) == 12 and len(meta_files) == 1, "資料不完整，請看 0.1"

# (3) 載入套件並印出版本
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import scipy, sklearn
import pyopenms as oms

for name, mod in [("pyopenms", oms), ("numpy", np), ("pandas", pd), ("scipy", scipy),
                  ("scikit-learn", sklearn), ("matplotlib", matplotlib), ("seaborn", sns)]:
    print(f"{name:13s} {mod.__version__}")
print("Python", sys.version.split()[0], "| Colab" if IN_COLAB else "| 本機")

### 0.1 如果 pip 安裝或下載失敗怎麼辦

| 狀況 | 怎麼辦 |
|---|---|
| `pip` 顯示 `No matching distribution found for pyopenms==3.6.0` | Python 版本和套件不合。執行 `!python --version`，把結果告訴老師或助教。 |
| `pip` 跑完後出現「需要重新啟動執行階段（Restart session）」 | 按「執行階段 → 重新啟動工作階段」，然後**從第 0 節再執行一次**（已安裝的不會重裝）。 |
| 下載時顯示 `HTTP Error`（例如 404、429）或連線失敗 | 用瀏覽器打開 https://github.com/wang3yuan/PBAIH/releases/tag/data-v1 ，手動下載 `sacurine_cropped.zip`，在 Colab 左側「檔案」面板上傳，再執行下面「手動放資料」那段程式。 |
| 網路逾時、下載到一半中斷 | 刪掉 `pbaih11_data` 資料夾後再執行一次。 |
| GitHub 也打不開（最後手段） | 從 Zenodo record 10130758（CC-BY-4.0）下載原始的 Sacurine toy set。**原始檔沒有裁切、全部下載約需 17 分鐘**，而且之後的數字會和本 notebook 不同，所以只當備援。 |

手動放資料（上傳 zip 之後）：

```python
import zipfile, os
zipfile.ZipFile("sacurine_cropped.zip").extractall("pbaih11_data")
# 或者，若資料放在你自己的 Google 雲端硬碟：
# from google.colab import drive; drive.mount("/content/drive")
# os.environ["PBAIH11_DATA_DIR"] = "/content/drive/MyDrive/<你放 mzML 的資料夾>"
```
然後重新執行第 0 節的程式格。

**求救順序**：① 讀錯誤訊息最後一行 → ② 問 Colab 的 Gemini → ③ 打開提示 → ④ 舉手問助教／老師。

## 1. 讀一個 mzML：TIC／BPC 與 MS1 譜圖

**為什麼**：在做任何演算法之前，先「看一眼」原始資料：這個檔案有幾張譜圖？是 centroid 還是 profile？層析圖（TIC／BPC）長怎樣？PBAIH07 我們用 `MSExperiment` 讀過 mzML，今天換成真實的代謝體檔案。

- **TIC**（total ion chromatogram）：每張譜圖所有峰的強度**加總**，對保留時間（RT）作圖。
- **BPC**（base peak chromatogram）：每張譜圖中**最高**那個峰的強度，對 RT 作圖。BPC 比較不受大量小雜訊峰影響，峰形較清楚。

**怎麼做**：先讀樣本中繼資料表（哪個檔是樣本、QC、blank，注入順序），再讀一個 QC 檔。

In [ ]:
# 1-1 樣本中繼資料（sample metadata）
meta = pd.read_csv(meta_files[0], sep="\t").set_index("sample_name")
meta = meta.sort_values(["sampleType", "injectionOrder"])
meta["file"] = [os.path.join(DATA_DIR, f"{s}.mzML") for s in meta.index]
display(meta[["class", "sampleType", "injectionOrder", "batch", "gender", "age"]])

In [ ]:
# 1-2 讀一個 QC 檔（接續 PBAIH07：MSExperiment + MzMLFile().load）
DEMO_SAMPLE = "QC1_008"
exp = oms.MSExperiment()
oms.MzMLFile().load(meta.loc[DEMO_SAMPLE, "file"], exp)
exp.sortSpectra(True)   # 依 RT 排序；後面的峰偵測需要排序過的資料

first = exp[0]
print("譜圖數：", exp.getNrSpectra())
print("MS 層級：", sorted({s.getMSLevel() for s in exp}))
print(f"RT 範圍：{exp[0].getRT():.1f} – {exp[exp.getNrSpectra()-1].getRT():.1f} 秒")
print("極性：", first.getInstrumentSettings().getPolarity())
print("譜圖型態：", first.getType())
is_centroid = first.getType() == oms.SpectrumSettings.SpectrumType.CENTROID
print("已經是 centroid 嗎？", is_centroid)

**看結果**：這個檔只有 MS1（沒有 MS2），而且已經是 **centroid**——每個離子只用一個 m/z 值表示（PBAIH07 做過 `PeakPickerHiRes` 的那一步，這裡廠商軟體已經做好）。後面的 `MassTraceDetection` **只接受 centroid 資料**，如果是 profile 就要先做 centroiding。

### 練習 1：算出 TIC 與 BPC

In [ ]:
# 1-3 計算 TIC 與 BPC
rts, tic, bpc = [], [], []
for spec in exp:
    mz, inten = spec.get_peaks()      # 一張譜圖的 m/z 與強度（numpy 陣列）
    rts.append(spec.getRT())
    tic.append(inten.sum())           # TIC：全部加總
    bpc.append(inten.max())           # 練習 1
rts, tic, bpc = np.array(rts), np.array(tic), np.array(bpc)

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(rts, tic, lw=0.8, color="#0F8B8D"); axes[0].set_ylabel("TIC intensity")
axes[1].plot(rts, bpc, lw=0.8, color="#EE6C4D"); axes[1].set_ylabel("BPC intensity")
axes[1].set_xlabel("Retention time (s)")
axes[0].set_title(f"{DEMO_SAMPLE}: TIC and BPC")
plt.tight_layout(); plt.show()

<details><summary>💡 練習 1 提示（先自己試，卡住再打開）</summary>

TIC 用 `inten.sum()`；BPC 要的是最大值。numpy 陣列有 `.max()` 方法。

<details><summary>參考答案</summary>

```python
bpc.append(inten.max())
```
</details>
</details>

In [ ]:
# 1-4 看 BPC 最高點那一張 MS1 譜圖（centroid：每個峰是一條直線）
i_top = int(np.argmax(bpc))
mz, inten = exp[i_top].get_peaks()
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.vlines(mz, 0, inten, lw=0.8, color="#1A2238")
j = int(np.argmax(inten))
ax.annotate(f"m/z {mz[j]:.4f}", (mz[j], inten[j]), xytext=(10, -10), textcoords="offset points")
ax.set_xlabel("m/z"); ax.set_ylabel("Intensity")
ax.set_title(f"MS1 spectrum at RT = {rts[i_top]:.1f} s ({len(mz)} centroid peaks)")
plt.tight_layout(); plt.show()

all_int = np.concatenate([s.get_peaks()[1] for s in exp])
print(f"這個檔所有峰的強度：中位數 {np.median(all_int):.0f}，第 90 百分位 {np.percentile(all_int, 90):.0f}")

**想一想**
1. TIC 和 BPC 哪一條的峰比較「尖」？為什麼？
2. 上面印出的強度中位數只有幾千。等一下設定「雜訊門檻」時，如果設成 10000，會發生什麼事？

## 2. 單一檔案的特徵偵測：質量軌跡 → 洗脫峰 → 特徵

**為什麼**：一個代謝物離子會在連續好幾張 MS1 譜圖中、在幾乎相同的 m/z 出現，形成一條「**質量軌跡**（mass trace）」。把軌跡沿 RT 切成一個個**洗脫峰**，再把同一個分子的同位素峰（¹²C、¹³C…）組在一起，就得到一個**特徵（feature）**＝（m/z, RT, 強度）。這就是講課中的步驟②峰偵測。

pyopenms 的三步驟（和 OpenMS 的 `FeatureFinderMetabo` 工具相同）：

| 步驟 | 類別 | 關鍵參數 | 意義 |
|---|---|---|---|
| 2-1 | `MassTraceDetection` | `mass_error_ppm` | 同一條軌跡相鄰掃描之間 m/z 可容許的差距（ppm）。Orbitrap 常用 5–10。 |
| | | `noise_threshold_int` | 低於這個強度的峰不拿來起始軌跡。pyopenms 預設是 10；這批資料強度中位數只有幾千，我們用 **1000**。 |
| 2-2 | `ElutionPeakDetection` | `width_filtering` | 依峰寬（FWHM）過濾；`auto` 讓程式依資料估計合理範圍。 |
| 2-3 | `FeatureFindingMetabo` | `remove_single_traces` | 只有一條軌跡（找不到同位素峰）的就刪掉。`true` 可大幅減少雜訊特徵，但也會刪掉低強度的真實訊號。 |
| | | `charge_lower_bound`／`charge_upper_bound` | 同位素間距用來推電荷；小分子幾乎都是 1 價（預設 1–3）。 |

### 2-1 質量軌跡偵測（練習 2）

In [ ]:
# 2-1 MassTraceDetection：把各張譜圖中 m/z 相近、RT 連續的峰串成軌跡
PPM = 10.0
NOISE = 1e3

mtd = oms.MassTraceDetection()
mtd.setLogType(oms.LogType.NONE)          # 關掉進度列，輸出比較乾淨
params = mtd.getDefaults()
params.setValue("mass_error_ppm", PPM)
params.setValue("noise_threshold_int", NOISE)   # 練習 2
mtd.setParameters(params)

traces = mtd.run(exp, 0)                  # pyopenms 3.6：直接回傳 list（0 = 不限制軌跡數）
print("質量軌跡數：", len(traces))

<details><summary>💡 練習 2 提示（先自己試，卡住再打開）</summary>

參數名稱是 `noise_threshold_int`，值用上面已經定義好的變數（不要直接打數字，之後才方便一起改）。

<details><summary>參考答案</summary>

```python
params.setValue("noise_threshold_int", NOISE)
```
</details>
</details>

In [ ]:
# 2-2 ElutionPeakDetection：一條長軌跡裡可能有好幾個層析峰，把它切開；再依峰寬過濾
epd = oms.ElutionPeakDetection()
epd.setLogType(oms.LogType.NONE)
params = epd.getDefaults()
params.setValue("width_filtering", "auto")
epd.setParameters(params)

peaks = epd.detectPeaks(traces)           # 回傳 list
peaks = epd.filterByPeakWidth(peaks)      # 回傳 list
print("洗脫峰數：", len(peaks))

# 2-3 FeatureFindingMetabo：把同一分子的同位素軌跡組成特徵
ffm = oms.FeatureFindingMetabo()
ffm.setLogType(oms.LogType.NONE)
params = ffm.getDefaults()
params.setValue("remove_single_traces", "true")
params.setValue("report_convex_hulls", "true")   # 保留每個特徵的 RT 範圍，畫圖用
ffm.setParameters(params)

fmap = oms.FeatureMap()
ffm.run(peaks, fmap)                      # 結果寫進 fmap（pyopenms 3.6 另外回傳 tuple，可忽略）
fmap.setUniqueIds()
print("特徵數：", fmap.size())

In [ ]:
# 2-4 把 FeatureMap 整理成 DataFrame
def features_to_df(fmap):
    rows = []
    for f in fmap:
        hull = f.getConvexHulls()[0].getBoundingBox()    # 第一條（單一同位素）軌跡的範圍
        rows.append({
            "mz": f.getMZ(), "rt": f.getRT(), "intensity": f.getIntensity(),
            "n_traces": int(f.getMetaValue("num_of_masstraces")),
            "fwhm_s": f.getMetaValue("FWHM"),
            "rt_start": hull.minPosition()[0], "rt_end": hull.maxPosition()[0],
            "trace_mz": list(f.getMetaValue("masstrace_centroid_mz")),
        })
    return pd.DataFrame(rows).sort_values("intensity", ascending=False).reset_index(drop=True)

feat = features_to_df(fmap)
print("每個特徵含幾條同位素軌跡：", feat["n_traces"].value_counts().sort_index().to_dict())
display(feat.drop(columns="trace_mz").head(8).round(4))

**看結果**：從約 3,500 條質量軌跡，到約 5,000 個洗脫峰，最後只剩約 200 多個特徵——因為 `remove_single_traces=true` 把找不到同位素夥伴的訊號全部刪掉了。`intensity` 是單一同位素那條軌跡的**峰面積**（不是峰高）。

### 2-5 畫一個特徵的 EIC

**EIC**（extracted ion chromatogram）：只取某個 m/z 視窗（±10 ppm）內的強度，對 RT 作圖。下面挑一個「有同位素、峰寬合理」的強特徵，把 ¹²C 與 ¹³C 兩條 EIC 疊在一起，並標出演算法判定的峰範圍。

In [ ]:
def extract_eic(exp, target_mz, ppm=10.0):
    """回傳 (rt, intensity)：每張譜圖在 target_mz ± ppm 內的最大強度。"""
    tol = target_mz * ppm * 1e-6
    rt, y = [], []
    for spec in exp:
        mz, inten = spec.get_peaks()
        lo, hi = np.searchsorted(mz, [target_mz - tol, target_mz + tol])
        rt.append(spec.getRT())
        y.append(inten[lo:hi].max() if hi > lo else 0.0)
    return np.array(rt), np.array(y)

# 挑選示範特徵：至少 2 條同位素軌跡、FWHM 2–15 秒、峰範圍 < 90 秒，取強度最高者
cand = feat[(feat.n_traces >= 2) & feat.fwhm_s.between(2, 15) & ((feat.rt_end - feat.rt_start) < 90)]
demo = cand.iloc[0]          # 想看別的特徵？把 0 改成 1、2、3…
print(demo[["mz", "rt", "intensity", "n_traces", "fwhm_s"]])

win = (demo.rt - 60, demo.rt + 60)
fig, ax = plt.subplots(figsize=(10, 4))
for k, (lab, col) in enumerate([("monoisotopic (12C)", "#0F8B8D"), ("13C isotope", "#F2B134")]):
    if k < len(demo.trace_mz):
        r, y = extract_eic(exp, demo.trace_mz[k])
        m = (r > win[0]) & (r < win[1])
        ax.plot(r[m], y[m], color=col, lw=1.5, label=f"{lab}  m/z {demo.trace_mz[k]:.4f}")
ax.axvspan(demo.rt_start, demo.rt_end, color="#8D99AE", alpha=0.2, label="detected peak range")
ax.axvline(demo.rt, color="#EE6C4D", ls="--", lw=1, label=f"apex RT {demo.rt:.1f} s")
ax.set_xlabel("Retention time (s)"); ax.set_ylabel("Intensity")
ax.set_title(f"EIC of feature m/z {demo.mz:.4f} @ {demo.rt:.1f} s  (area = {demo.intensity:.3g})")
ax.legend(frameon=False); plt.tight_layout(); plt.show()

**想一想**
1. ¹³C 那條 EIC 的形狀和 ¹²C 幾乎一樣、高度較低——為什麼這可以當作「這是真的分子訊號」的證據？
2. 一個低濃度的真實代謝物，¹³C 峰可能低於雜訊門檻。`remove_single_traces=true` 會怎麼處理它？

## 3. 一次跑完 12 個檔案

**為什麼**：特徵表需要每個樣本都做一樣的峰偵測，參數也必須**全部相同**（並寫進方法）。

**怎麼做**：下面把第 2 節的三步驟包成現成函式 `detect_features()`，再用 `run_all()` 跑過全部檔案（附進度）。你只需要改參數、觀察特徵數怎麼變。

In [ ]:
def detect_features(path, ppm=10.0, noise=1e3, remove_single_traces=True):
    """一個 mzML → FeatureMap（第 2 節三步驟的濃縮版）。"""
    e = oms.MSExperiment(); oms.MzMLFile().load(path, e); e.sortSpectra(True)
    mtd = oms.MassTraceDetection(); mtd.setLogType(oms.LogType.NONE)
    p = mtd.getDefaults(); p.setValue("mass_error_ppm", float(ppm)); p.setValue("noise_threshold_int", float(noise))
    mtd.setParameters(p)
    epd = oms.ElutionPeakDetection(); epd.setLogType(oms.LogType.NONE)
    p = epd.getDefaults(); p.setValue("width_filtering", "auto"); epd.setParameters(p)
    pk = epd.filterByPeakWidth(epd.detectPeaks(mtd.run(e, 0)))
    ffm = oms.FeatureFindingMetabo(); ffm.setLogType(oms.LogType.NONE)
    p = ffm.getDefaults(); p.setValue("remove_single_traces", "true" if remove_single_traces else "false")
    p.setValue("report_convex_hulls", "true"); ffm.setParameters(p)
    fm = oms.FeatureMap(); ffm.run(pk, fm); fm.setUniqueIds()
    return fm

def run_all(meta, ppm=10.0, noise=1e3, remove_single_traces=True, verbose=True):
    """對 meta 中每個檔案跑 detect_features；回傳 (maps 清單, 摘要表)。"""
    maps, rows, t0 = [], [], time.time()
    for k, (name, row) in enumerate(meta.iterrows(), 1):
        t = time.time()
        fm = detect_features(row["file"], ppm, noise, remove_single_traces)
        maps.append(fm)
        rows.append({"sample": name, "type": row["sampleType"], "n_features": fm.size()})
        if verbose:
            print(f"[{k:2d}/{len(meta)}] {name:11s} {row['sampleType']:6s} 特徵數 {fm.size():4d}  ({time.time()-t:.1f} s)")
    if verbose:
        print(f"全部完成，共 {time.time()-t0:.1f} 秒")
    return maps, pd.DataFrame(rows)

maps, summary = run_all(meta, ppm=10.0, noise=1e3)

### 練習 3：把雜訊門檻調高 10 倍，特徵數怎麼變？

In [ ]:
NOISE_HIGH = 1e4   # 練習 3
maps_hi, summary_hi = run_all(meta, ppm=10.0, noise=NOISE_HIGH, verbose=False)

compare = summary.assign(n_features_high_noise=summary_hi["n_features"].values)
display(compare)

colors = {"pool": "#0F8B8D", "sample": "#6C5CE7", "blank": "#8D99AE"}
fig, ax = plt.subplots(figsize=(11, 3.8))
x = np.arange(len(compare))
ax.bar(x - 0.2, compare.n_features, 0.4, color=[colors[t] for t in compare.type], label="noise = 1e3")
ax.bar(x + 0.2, compare.n_features_high_noise, 0.4, color="#EE6C4D", alpha=0.7, label=f"noise = {NOISE_HIGH:g}")
ax.set_xticks(x, compare["sample"], rotation=45, ha="right")
ax.set_ylabel("Number of features"); ax.legend(frameon=False)
ax.set_title("Features per file (bar color = sample type for noise 1e3)")
plt.tight_layout(); plt.show()

<details><summary>💡 練習 3 提示（先自己試，卡住再打開）</summary>

1e3 = 1000；高 10 倍就是 10000，Python 可以寫成 `1e4`。也可以試試 `3e3`。

<details><summary>參考答案</summary>

```python
NOISE_HIGH = 1e4
```
</details>
</details>

**看結果**：blank 只有個位數到十幾個特徵，樣本與 QC 約 150–320 個；雜訊門檻調高 10 倍後，每個檔的特徵數大約剩一半以下。

**想一想**
1. 特徵數「越多越好」嗎？多出來的是什麼？少掉的又是什麼？
2. 為什麼 blank 的特徵數這麼少？這對後面的 blank 過濾和對齊有什麼影響？

## 4. 跨樣本對應：做成特徵表

**為什麼**：每個檔案各自有一串特徵，但「樣本 A 的 m/z 187.0068 @ 379 s」和「樣本 B 的 m/z 187.0069 @ 380 s」是不是同一個東西？**對應（correspondence）**就是把各檔中 m/z 與 RT 都夠接近的特徵配成同一列，得到「特徵 × 樣本」的矩陣（講課的步驟④）。

**怎麼做**：用 `FeatureGroupingAlgorithmKD`。重要的預設參數：`link:mz_tol` = 10 ppm、`link:rt_tol` = 30 秒；它內建一個簡單的 RT 校正（`warp:enabled` = true），所以小幅 RT 漂移在這一步就能被容忍（細節見延伸 E1）。某個樣本沒有對應的特徵 → 那一格就是**缺值（NaN）**。

> 執行時可能看到 `Warning: Only 3 data points for LOWESS fit of map 0…`：這是 KD 內建的 RT 校正在 blank（特徵很少）上找不到足夠的點，會改用較簡單的方式處理，可以忽略。

In [ ]:
def group_features(maps, names):
    """FeatureMap 清單 → (特徵表：列=特徵、欄=樣本, 特徵資訊表)。"""
    cmap = oms.ConsensusMap()
    grouper = oms.FeatureGroupingAlgorithmKD(); grouper.setLogType(oms.LogType.NONE)
    grouper.group(maps, cmap)
    rows = []
    for c in cmap:
        r = {"mz": c.getMZ(), "rt": c.getRT()}
        for h in c.getFeatureList():
            r[names[h.getMapIndex()]] = h.getIntensity()
        rows.append(r)
    df = pd.DataFrame(rows).reindex(columns=["mz", "rt"] + list(names))
    df = df.sort_values(["rt", "mz"]).reset_index(drop=True)
    ids = pd.Series([f"{mz:.4f}@{rt:.1f}" for mz, rt in zip(df.mz, df.rt)])   # 特徵命名 m/z@RT(秒)
    dup = ids.groupby(ids).cumcount()                                         # 名稱重複時加 _2、_3
    df.index = [i if k == 0 else f"{i}_{k+1}" for i, k in zip(ids, dup)]
    df.index.name = "feature"
    info = df[["mz", "rt"]].copy()
    info["n_detected"] = df[list(names)].notna().sum(axis=1)
    return df[list(names)], info

names = list(meta.index)
X, feat_info = group_features(maps, names)
print("特徵表大小（特徵 × 樣本）：", X.shape)
print("特徵名稱是否唯一：", X.index.is_unique)
display(X.head(8).round(0))

In [ ]:
# 4-2 檢視特徵表：每檔特徵數、缺值比例、每個特徵在幾個檔被偵測到
is_blank = meta["sampleType"].eq("blank")
not_blank = meta.index[~is_blank]
print(f"全部 12 欄的缺值比例：{X.isna().mean().mean():.1%}")
print(f"只看 9 個非 blank 欄：{X[not_blank].isna().mean().mean():.1%}")
print(f"在 9 個非 blank 中至少一半（≥5）被偵測到的特徵：{(X[not_blank].notna().sum(axis=1) >= 5).sum()}")

per_file = pd.DataFrame({"type": meta.sampleType, "detected": X.notna().sum(), "missing_%": (X.isna().mean() * 100).round(1)})
display(per_file)

fig, ax = plt.subplots(figsize=(8, 3.5))
vc = feat_info["n_detected"].value_counts().sort_index()
ax.bar(vc.index, vc.values, color="#0F8B8D")
ax.set_xlabel("Number of files in which the feature was detected"); ax.set_ylabel("Number of features")
ax.set_xticks(range(1, 13)); plt.tight_layout(); plt.show()

In [ ]:
# 4-3 特徵表要和樣本中繼資料表「欄對欄」一起保存
assert list(X.columns) == list(meta.index)
display(meta[["class", "sampleType", "injectionOrder", "batch"]].T)

**看結果**：12 個檔各約 5–320 個特徵，合併後變成六百多列；矩陣中大多數格子是缺值，且很多特徵只在 1 個檔中出現。

**想一想**
1. 缺值是「這個樣本真的沒有這個分子」，還是「有，但低於閾值／沒對應上」？你能從這張表分辨嗎？（實作二會再談）
2. 如果把 `link:rt_tol` 從 30 秒改成 5 秒，你預期特徵列數會變多還是變少？

## 5. Blank 過濾（練習 4）

**為什麼**：blank（只打溶劑、沒有樣本）裡出現的訊號，多半來自溶劑、管柱或環境污染，不是生物訊號。常見規則：**樣本平均強度 ÷ blank 平均強度 < 閾值**就刪掉該特徵。閾值（例如 3、5、10）是**經驗值，沒有唯一標準**，所以寫成參數，並記錄刪掉多少。

**怎麼做**：這裡把缺值當作 0（沒偵測到 ≈ 很低）。blank 完全沒偵測到的特徵，比值視為無限大（保留）。

In [ ]:
BLANK_RATIO = 3          # 閾值：可改成 5 或 10 看看

samples = meta.index[meta.sampleType.eq("sample")]
blanks = meta.index[meta.sampleType.eq("blank")]
sample_mean = X[samples].fillna(0).mean(axis=1)
blank_mean = X[blanks].fillna(0).mean(axis=1)
ratio = pd.Series(np.where(blank_mean > 0, sample_mean / blank_mean.where(blank_mean > 0), np.inf), index=X.index)

keep = ratio >= BLANK_RATIO      # 練習 4
X_filt = X[keep]

print(f"在任一 blank 中偵測到的特徵：{(blank_mean > 0).sum()}")
print(f"閾值 {BLANK_RATIO}：刪掉 {(~keep).sum()} 個，保留 {keep.sum()} 個")
display(pd.DataFrame({"sample_mean": sample_mean, "blank_mean": blank_mean, "ratio": ratio})[~keep].round(2))

<details><summary>💡 練習 4 提示（先自己試，卡住再打開）</summary>

`ratio` 是每個特徵的比值（一個 pandas Series），`BLANK_RATIO` 是閾值。Series 和數字比較會得到 True/False 的 Series。

<details><summary>參考答案</summary>

```python
keep = ratio >= BLANK_RATIO
```
</details>
</details>

**想一想**
1. 被刪掉的特徵裡，有沒有 blank 強度和樣本差不多的？那可能是什麼？
2. 我們只有 3 個 blank，而且 blank 特徵很少。把缺值當 0 的做法，什麼時候會出錯？
3. 為什麼在比值的分子用 `sample`（受試者樣本）而不是 QC？

🎉 **核心路徑完成！** 你已經把 12 個 mzML 變成一張過濾過 blank 的特徵表 `X_filt`。實作二會從「特徵表」接著做 QC 過濾、缺值、批次校正與正規化。以下延伸內容請課後或有餘力時再做。

---
# 延伸（課後／有餘力）

## E1. 保留時間對齊：執行現成程式並解讀

**為什麼**：同一化合物在不同樣本的 RT 會漂移（管柱老化、溫度、梯度）。對齊（alignment）先估計每個樣本相對於參考樣本的 RT 偏差，把 RT 校正回來，對應才不會配錯（講課步驟③）。

**怎麼做**：`MapAlignmentAlgorithmPoseClustering` 以特徵最多的樣本為參考，把其他樣本的特徵和參考配對，預設用配對點配一條**線性** RT 轉換；我們再用同一批配對點改配**LOWESS**（局部加權平滑，可處理非線性漂移）比較。**只用 9 個非 blank 檔**：blank 只有 5–11 個特徵，和參考只配得到約 2 對點，估不出可靠的轉換（資料偵查時也曾在 blank 上遇到 `no data points for linear model` 錯誤）。

下面的圖：每個點是一對「樣本特徵 ↔ 參考特徵」，縱軸是 RT 差（參考 − 樣本），越接近 0 越好。三格分別是對齊前、線性校正後、LOWESS 校正後。

In [ ]:
idx_nb = [i for i, n in enumerate(names) if meta.loc[n, "sampleType"] != "blank"]
maps_al = [oms.FeatureMap(maps[i]) for i in idx_nb]          # 複製一份，不改動原本的 maps
names_al = [names[i] for i in idx_nb]
ref = int(np.argmax([m.size() for m in maps_al]))
print("參考樣本：", names_al[ref])

aligner = oms.MapAlignmentAlgorithmPoseClustering(); aligner.setLogType(oms.LogType.NONE)
aligner.setReference(maps_al[ref])
dev_rows = []
for i, m in enumerate(maps_al):
    if i == ref:
        continue
    trafo = aligner.align(m)                                  # TransformationDescription（線性模型）
    pts = trafo.getDataPoints()                               # 用來配模型的配對點
    x = np.array([p.first for p in pts]); y = np.array([p.second for p in pts])
    x_lin = np.array([trafo.apply(v) for v in x])
    trafo_lo = oms.TransformationDescription(trafo)
    trafo_lo.fitModel("lowess", oms.Param())                  # 同一批點改配 LOWESS
    x_lo = np.array([trafo_lo.apply(v) for v in x])
    oms.MapAlignmentTransformer().transformRetentionTimes(m, trafo_lo, True)
    for a, b, c, d in zip(x, y, x_lin, x_lo):
        dev_rows.append({"sample": names_al[i], "rt": a, "before": b - a, "linear": b - c, "lowess": b - d})
dev = pd.DataFrame(dev_rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, col in zip(axes, ["before", "linear", "lowess"]):
    for s, d in dev.groupby("sample"):
        ax.scatter(d.rt, d[col], s=8, alpha=0.6, label=s)
    ax.axhline(0, color="k", lw=0.8); ax.set_ylim(-15, 15)
    ax.set_title(f"{col}: median |dRT| = {dev[col].abs().median():.2f} s")
    ax.set_xlabel("Retention time (s)")
axes[0].set_ylabel("RT(reference) - RT(sample) (s)")
axes[2].legend(fontsize=7, frameon=False, ncol=2)
plt.tight_layout(); plt.show()

tbl = dev.groupby("sample")[["before", "linear", "lowess"]].agg(lambda v: np.median(np.abs(v))).round(2)
tbl["n_pairs"] = dev.groupby("sample").size()
print("各樣本 |ΔRT| 中位數（秒）："); display(tbl)
early = dev.rt.between(150, 230)
print("150–230 秒區段 ΔRT 中位數（秒）：", dev[early][["before", "linear", "lowess"]].median().round(2).to_dict())
print(f"參考：MS1 掃描間隔約 {np.median(np.diff(rts)):.2f} 秒；|ΔRT| > 15 秒的配對點（多半是配錯）有 {(dev.before.abs() > 15).sum()} 個")

In [ ]:
# 對齊後再做一次對應，和第 4 節（未另外對齊）比較
X_al, info_al = group_features(maps_al, names_al)
X_nb, _ = group_features([maps[i] for i in idx_nb], names_al)
print(f"9 個非 blank 檔，未另外對齊：{X_nb.shape[0]} 個特徵，缺值 {X_nb.isna().mean().mean():.1%}")
print(f"9 個非 blank 檔，先 LOWESS 對齊再對應：{X_al.shape[0]} 個特徵，缺值 {X_al.isna().mean().mean():.1%}")

**解讀**（數字以你跑出來的為準）：
- 這批資料的 RT 漂移本來就很小：多數配對的偏差在 1 秒左右，約 1–2 個掃描間隔。
- **線性**校正後，|ΔRT| 中位數**沒有變小**，多數樣本反而略增；**LOWESS** 只有小幅改善。
- 看左圖會發現漂移不是一條直線：約 150–230 秒有一段系統性的負偏差（中位數約 −2.7 秒），其他區段接近 0。而且**每個樣本在這段都往同一個方向偏**，代表很可能是**參考樣本自己**在這段偏了——參考樣本怎麼選也會影響結果。
- 直線模型修不了這種局部彎曲，還會被少數配錯的點（離 0 很遠的點）拉偏；LOWESS 用預設的平滑範圍（span ≈ 2/3）也太寬，這段偏差只從約 −2.7 秒縮到約 −1.9 秒。
- 先對齊再對應，得到的特徵表和不另外對齊幾乎一樣：KD 對應本身就容許 30 秒內的差距，也內建 RT 校正。
- 結論：**對齊不是每次都會帶來明顯改善**。漂移小時效益有限；漂移大或呈非線性時，就要選對模型。

**想一想**
1. 什麼樣的實驗設計會讓 RT 漂移變大、對齊變得必要？（提示：跨好幾天、換管柱、樣本數上千）
2. 為什麼「把對齊前後的偏差畫出來看」比只看一個中位數更重要？

## E2. QC RSD 預覽（選做練習 5）

**為什麼**：pooled QC 是把所有樣本混在一起、重複打很多次的「同一個樣本」。同一個特徵在 QC 之間的變異（RSD＝標準差 ÷ 平均 × 100%）反映**量測本身**的穩定度。常見經驗門檻是 RSD 20–30%（**非定論**，實作二會再用）。

In [ ]:
qcs = meta.index[meta.sampleType.eq("pool")]
in_all_qc = X[qcs].notna().all(axis=1)
Q = X.loc[in_all_qc, qcs]
rsd = Q.std(axis=1) / Q.mean(axis=1) * 100     # 練習 5（選做）

print(f"3 個 QC 都偵測到的特徵：{in_all_qc.sum()} 個；RSD 中位數 {rsd.median():.1f}%；RSD < 30% 的比例 {(rsd < 30).mean():.0%}")
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(rsd, bins=30, color="#0F8B8D")
ax.axvline(30, color="#EE6C4D", ls="--", label="30% (common rule of thumb)")
ax.set_xlabel("QC RSD (%)"); ax.set_ylabel("Number of features"); ax.legend(frameon=False)
plt.tight_layout(); plt.show()

<details><summary>💡 練習 5 提示（先自己試，卡住再打開）</summary>

對每一列（每個特徵）跨欄計算：`Q.std(axis=1)`、`Q.mean(axis=1)`。

<details><summary>參考答案</summary>

```python
rsd = Q.std(axis=1) / Q.mean(axis=1) * 100
```
</details>
</details>

**想一想**：只有 3 個 QC 估計出的 RSD 可靠嗎？這裡的 QC 已經做過任何校正了嗎？

## E3. 存檔與參數敏感度

**為什麼**：前處理的結果要連同**參數**一起保存，別人（或兩個月後的你）才能重現。下面把特徵表、中繼資料與參數存成 CSV／JSON，並看看兩個關鍵參數如何改變單一檔案的特徵數。

In [ ]:
import json
X_filt.to_csv("PBAIH11A_feature_table.csv")
feat_info.loc[X_filt.index].to_csv("PBAIH11A_feature_info.csv")
meta.drop(columns="file").to_csv("PBAIH11A_sample_metadata.csv")
run_params = {"pyopenms": oms.__version__, "mass_error_ppm": 10.0, "noise_threshold_int": 1e3,
              "width_filtering": "auto", "remove_single_traces": True,
              "grouping": "FeatureGroupingAlgorithmKD (defaults)", "blank_ratio": BLANK_RATIO}
json.dump(run_params, open("PBAIH11A_params.json", "w"), indent=2)
print("已存：PBAIH11A_feature_table.csv", X_filt.shape, "、feature_info、sample_metadata、params.json")

In [ ]:
# 參數敏感度：同一個 QC 檔，改雜訊門檻與 remove_single_traces
path = meta.loc[DEMO_SAMPLE, "file"]
grid = []
for noise in [1e2, 1e3, 3e3, 1e4]:
    for rst in [True, False]:
        grid.append({"noise": noise, "remove_single_traces": rst,
                     "n_features": detect_features(path, 10.0, noise, rst).size()})
for ppm in [5.0, 20.0]:
    grid.append({"noise": 1e3, "remove_single_traces": True, "ppm": ppm,
                 "n_features": detect_features(path, ppm, 1e3, True).size()})
sens = pd.DataFrame(grid).fillna({"ppm": 10.0})
display(sens.pivot_table(index=["ppm", "noise"], columns="remove_single_traces", values="n_features"))

**想一想**
1. `remove_single_traces` 和雜訊門檻，哪一個對特徵數的影響比較大？
2. 雜訊門檻從 100 調到 1000，在 `remove_single_traces=true` 時特徵數幾乎沒變，為什麼？
3. 如果要替自己的資料選參數，你會用什麼當「標準答案」來比較？（提示：已知濃度的標準品、QC 的重現性）

---
**下一步**：實作二會用另一個有 pooled QC 與多個批次的公開特徵表，練習 QC／RSD／偵測率過濾、缺值、批次校正、正規化與轉換。